# 01 — Diagnosis Kualitas Data
### Milestone M2 · Exploratory Data Analysis (KMTI21133)

| | |
|---|---|
| **Nama tim** | Kelompok Bakery Sales |
| **Anggota** | Alfandi Panca Saputra (0110225026) <br>Muhammad Sultan Ridho (0110225099)<br>Dzakwan Abdur Rasyid (0110225036)<br>Mumtaz Abdurrahman (0110225164) |
| **Topik proyek** | Eksplorasi Data Penjualan Bakery |
| **Tanggal** | 9 November 2026 |

**Notebook ini mengisi:** dimensi **Kelengkapan diagnosis kualitas data** (bobot 25) pada Rubrik M2.

Tujuan notebook ini **hanya mendiagnosis** — belum memperbaiki apa pun. Pembersihan
dikerjakan di `02_cleaning.ipynb`.

In [2]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 60)
plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 10,
    'axes.titlesize': 11, 'axes.titleweight': 'bold',
    'axes.grid': True, 'grid.alpha': 0.25,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
})

DATA = '#0E7490'
SOROT = '#C2410C'
REDUP = '#5B6672'

AKAR = Path('.')
RAW = AKAR / 'data' / 'raw'
PROCESSED = AKAR / 'data' / 'processed'
FIGURES = AKAR / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

## 0. Muat data mentah

Baca dari `data/raw`. Berkas ini **tidak pernah diubah** sepanjang proyek.

In [4]:
RAW.mkdir(parents=True, exist_ok=True)
import shutil
from pathlib import Path

DRIVE_BERKAS = Path('/content/Bakery sales (2).csv')
BERKAS = RAW / 'Bakery sales (2).csv'

if not BERKAS.exists() and DRIVE_BERKAS.exists():
    shutil.copy(DRIVE_BERKAS, BERKAS)

source_path = BERKAS if BERKAS.exists() else DRIVE_BERKAS
df = pd.read_csv(source_path)
print(f'{df.shape[0]} baris x {df.shape[1]} kolom')
display(df.head())

234005 baris x 7 kolom


,Unnamed: 0,date,time,ticket_number,article,Quantity,unit_price
0,0,2021-01-02,08:38,150040.0,BAGUETTE,1.0,"0,90 €"
1,1,2021-01-02,08:38,150040.0,PAIN AU CHOCOLAT,3.0,"1,20 €"
2,4,2021-01-02,09:14,150041.0,PAIN AU CHOCOLAT,2.0,"1,20 €"
3,5,2021-01-02,09:14,150041.0,PAIN,1.0,"1,15 €"
4,8,2021-01-02,09:25,150042.0,TRADITIONAL BAGUETTE,5.0,"1,20 €"


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 234005 entries, 0 to 234004
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Unnamed: 0     234005 non-null  int64  
 1   date           234005 non-null  object 
 2   time           234005 non-null  object 
 3   ticket_number  234005 non-null  float64
 4   article        234005 non-null  object 
 5   Quantity       234005 non-null  float64
 6   unit_price     234005 non-null  object 
dtypes: float64(2), int64(1), object(4)
memory usage: 12.5+ MB


## 1. Kelengkapan — nilai yang hilang

In [ ]:
kosong = pd.DataFrame({
    'jumlah' : df.isna().sum(),
    'persen' : (df.isna().mean() * 100).round(2),
})
kosong = kosong[kosong.jumlah > 0].sort_values('jumlah', ascending=False)
print(kosong.to_string() if len(kosong) else 'Tidak ada nilai kosong pada seluruh kolom dataset ini.')

Tidak ada nilai kosong pada seluruh kolom dataset ini.


In [ ]:
# Karena tidak ada data kosong, pengecekan pola tidak menghasilkan temuan relevan
KOLOM_X = 'article'
PEMBANDING = 'Quantity'

pembanding = df.groupby(df[KOLOM_X].isna())[PEMBANDING].agg(['count','mean']).round(2)
pembanding.index = ['terisi'] if len(pembanding) == 1 else ['terisi', 'kosong']
print(pembanding.to_string())

         count  mean
terisi  234005  1.54


**Tulis temuan Anda di sini:**
> Dataset sangat lengkap (MCAR, MAR, MNAR tidak relevan) karena 100% sel terisi.

## 2. Keunikan — baris kembar

In [ ]:
print('Duplikat seluruh kolom :', df.duplicated().sum())

KUNCI = ['date', 'time', 'ticket_number', 'article', 'Quantity'] # transaksi spesifik
print('Duplikat pada kolom kunci:', df.duplicated(subset=KUNCI).sum())

Duplikat seluruh kolom : 0
Duplikat pada kolom kunci: 1406


## 3. Konsistensi — penulisan kategori

In [ ]:
kategorikal = df.select_dtypes(include=['object', 'string']).columns.tolist()

for k in kategorikal:
    n_mentah  = df[k].nunique()
    n_rapi    = df[k].astype(str).str.strip().str.lower().nunique()
    tanda = '  <-- PERIKSA' if n_rapi < n_mentah else ''
    print(f'{k:24s} unik: {n_mentah:4d}   setelah dirapikan: {n_rapi:4d}{tanda}')

date                     unik:  600   setelah dirapikan:  600
time                     unik:  683   setelah dirapikan:  683
article                  unik:  149   setelah dirapikan:  149
unit_price               unik:  123   setelah dirapikan:  123


In [ ]:
KOLOM = 'article'
print(f"Contoh unik {KOLOM}:", sorted(df[KOLOM].dropna().unique())[:5])

Contoh unik article: ['.', '12 MACARON', 'ARMORICAIN', 'ARTICLE 295', 'BAGUETTE']


## 4. Validitas — nilai yang mustahil dan tipe data

In [ ]:
print('TIPE DATA YANG PERLU DIPERIKSA')
for k in df.columns:
    if df[k].dtype == object:
        contoh = df[k].dropna().iloc[0] if df[k].notna().any() else ''
        print(f'  {k:24s} object   contoh: {contoh!r}')

TIPE DATA YANG PERLU DIPERIKSA
  date                     object   contoh: '2021-01-02'
  time                     object   contoh: '08:38'
  article                  object   contoh: 'BAGUETTE'
  unit_price               object   contoh: '0,90 €'


In [ ]:
# Rentang nilai numerik -- cari nilai yang tidak mungkin (negatif, nol, terlalu besar)
num = df.select_dtypes('number')
print(num.describe().T[['min','25%','50%','75%','max']].round(2).to_string())

                    min       25%       50%       75%       max
Unnamed: 0          0.0  127979.0  254573.0  382911.0  511395.0
ticket_number  150040.0  184754.0  218807.0  253927.0  288913.0
Quantity         -200.0       1.0       1.0       2.0     200.0


In [ ]:
print('Kuantitas negatif atau 0 (retur/batal):', (df['Quantity'] <= 0).sum())
print('Tanggal tidak valid (bukan ISO):', (~df['date'].str.match(r'^\d{4}-\d{2}-\d{2}$')).sum())

Kuantitas negatif atau 0 (retur/batal): 1295
Tanggal tidak valid (bukan ISO): 0


## 5. Akurasi — konsistensi antarkolom

In [ ]:
# Karena belum ada total, tidak ada konsistensi kalkulasi secara langsung saat ini
# Tapi kita bisa pastikan setiap nomor tiket terjadi pada satu tanggal yang sama
tiket_beda_hari = df.groupby('ticket_number')['date'].nunique()
print('Tiket yang terbelah ke beberapa hari:', (tiket_beda_hari > 1).sum())


Tiket yang terbelah ke beberapa hari: 0


## 6. Ringkasan temuan

| # | Masalah | Dimensi | Jumlah / bukti | Rencana penanganan |
|---|---------|---------|----------------|--------------------|
| 1 | `Unnamed: 0` berlebih | Relevansi | 234.005 baris | Drop kolom |
| 2 | `unit_price` adalah object | Validitas (tipe data) | format "1,20 €" | Konversi string -> ganti koma ke titik, hapus €, jadikan float |
| 3 | `Quantity` <= 0 | Validitas (range) | 1295 baris | Filter hapus baris <= 0 (asumsi dibatalkan/retur) atau diabaikan |
| 4 | Duplikat kolom kunci | Keunikan | 3583 baris duplikat kunci | Drop duplicates jika identik |
| 5 | Tipe `date` & `time` | Validitas | Object | Konversi jadi datetime untuk time series |